# ANN untuk Prediksi Pendapatan Pelanggan (Business Intelligence)

Notebook ini adalah awalan untuk mengolah data bisnis, melatih model Artificial Neural Network (ANN), mengevaluasi performa, dan menghasilkan insight manajerial.

Dataset yang digunakan (di repo yang sama): `ann_business_intelligence_dataset.csv`

## 1) Import library

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout


## 2) Load dataset

In [ ]:
DATA_PATH = Path('ann_business_intelligence_dataset.csv')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. Tambahkan file ke root repo terlebih dahulu.'
    )

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
display(df.head())
display(df.info())

## 3) Data cleaning & preprocessing

In [ ]:
target_col = 'Total_Revenue'
required_columns = [
    'Customer_ID',
    'Age',
    'City',
    'Number_of_Transactions',
    'Average_Transaction_Value',
    'Favorite_Product_Category',
    'Monthly_Purchase_Frequency',
    'Months_as_Customer',
    target_col,
]

missing_cols = [c for c in required_columns if c not in df.columns]
if missing_cols:
    raise ValueError(f'Kolom wajib tidak lengkap: {missing_cols}')

# Drop ID karena bukan sinyal perilaku bisnis
X = df.drop(columns=[target_col, 'Customer_ID']).copy()
y = df[target_col].copy()

# Tangani missing value sederhana (bisa Anda kembangkan)
for col in X.select_dtypes(include=['number']).columns:
    X[col] = X[col].fillna(X[col].median())

for col in X.select_dtypes(exclude=['number']).columns:
    mode_series = X[col].mode(dropna=True)
    fallback = 'Unknown' if mode_series.empty else mode_series.iloc[0]
    X[col] = X[col].fillna(fallback)

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

if hasattr(X_train_prep, 'toarray'):
    X_train_prep = X_train_prep.toarray()
    X_test_prep = X_test_prep.toarray()

print('Train shape (setelah preprocessing):', X_train_prep.shape)
print('Test shape  (setelah preprocessing):', X_test_prep.shape)

## 4) Bangun & latih model ANN (Keras/TensorFlow)

In [ ]:
tf.random.set_seed(42)

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_prep.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='linear')
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

history = model.fit(
    X_train_prep,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    verbose=0
)

## 5) Evaluasi model (MAE, RMSE, R²)

In [ ]:
y_pred = model.predict(X_test_prep).ravel()

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'MAE : {mae:,.2f}')
print(f'RMSE: {rmse:,.2f}')
print(f'R²  : {r2:.4f}')

## 6) Visualisasi hasil

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.6)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--')
plt.xlabel('Actual Total_Revenue')
plt.ylabel('Predicted Total_Revenue')
plt.title('Actual vs Predicted')
plt.grid(True)
plt.show()

residuals = y_test - y_pred
plt.figure(figsize=(7, 4))
plt.hist(residuals, bins=20)
plt.title('Distribusi Residual')
plt.xlabel('Error (Actual - Predicted)')
plt.ylabel('Frekuensi')
plt.grid(True)
plt.show()

## 7) Insight bisnis awal

In [ ]:
result = X_test.copy()
result['actual_revenue'] = y_test.values
result['predicted_revenue'] = y_pred
result['prediction_error'] = result['actual_revenue'] - result['predicted_revenue']

print('Top 5 pelanggan dengan prediksi revenue tertinggi (subset test):')
display(result.sort_values('predicted_revenue', ascending=False).head(5))

print('Rata-rata revenue prediksi per kota:')
display(result.groupby('City', as_index=False)['predicted_revenue'].mean().sort_values('predicted_revenue', ascending=False))